**生成式 AI 使用说明**：本作业中使用生成式 AI 工具时，适用与协作相同的课程政策。和其他协作者一样，每位学生都必须独立写出自己的解答，不能直接依赖交互输出；提交内容中还应注明协作的性质。使用生成式 AI 工具实质性完成作业内容不符合本作业的精神，也会违反 [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 将你的 Google Drive 挂载到 Colab VM。
from google.colab import drive
drive.mount('/content/drive')

# TODO：填写 Drive 中保存解压后
# 作业文件夹，例如 'cs231n/assignments/assignment1/'
FOLDERNAME = 'cs231n/assignments/assignment1/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# 现在已经挂载 Drive，下面确保
# Colab VM 的 Python 解释器可以加载
# 其中的 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 将 CIFAR-10 数据集下载到你的 Drive
# 如果它还不存在。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
# %cd /content/drive/My\ Drive/$FOLDERNAME
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# 多层全连接网络
在这个练习中，你将实现一个可以包含任意数量隐藏层的全连接网络。

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

阅读 `cs231n/classifiers/fc_net.py` 文件中的 `FullyConnectedNet` 类。

实现网络初始化、前向传播和反向传播。在整个作业中，你会在 `cs231n/layers.py` 中实现不同层。这里可以复用你之前实现的 `affine_forward`、`affine_backward`、`relu_forward`、`relu_backward` 和 `softmax_loss`。目前还不需要实现 dropout 或 batch/layer normalization；这些功能会在后续部分加入。

In [ ]:
# 设置单元。
import time
import numpy as np
import matplotlib.pyplot as plt
from cs231n.classifiers.fc_net import *
from cs231n.data_utils import get_CIFAR10_data
from cs231n.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from cs231n.solver import Solver

%matplotlib inline
plt.rcParams["figure.figsize"] = (10.0, 8.0)  # 设置图像的默认大小。
plt.rcParams["image.interpolation"] = "nearest"
plt.rcParams["image.cmap"] = "gray"

%load_ext autoreload
%autoreload 2

def rel_error(x, y):
    """Returns relative error."""
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [ ]:
# 加载预处理后的 CIFAR-10 数据。
data = get_CIFAR10_data()
for k, v in list(data.items()):
    print(f"{k}: {v.shape}")

## 初始损失与梯度检查

作为合理性检查，运行下面的代码来检查初始损失，并分别在有正则化和无正则化的情况下对网络做梯度检查。这有助于判断初始损失是否在合理范围内。

做梯度检查时，你应该看到大约 `1e-7` 或更小的误差。

In [ ]:
np.random.seed(231)
N, D, H1, H2, C = 2, 15, 20, 30, 10
X = np.random.randn(N, D)
y = np.random.randint(C, size=(N,))

for reg in [0, 3.14]:
    print("Running check with reg = ", reg)
    model = FullyConnectedNet(
        [H1, H2],
        input_dim=D,
        num_classes=C,
        reg=reg,
        weight_scale=5e-2,
        dtype=np.float64
    )

    loss, grads = model.loss(X, y)
    print("Initial loss: ", loss)

    # 大多数误差应该在 e-7 或更小的量级。
    # 注意：当 reg = 0.0 时，W2 的误差在 e-5 量级也是可以接受的。
    # 这是该检查中的正常现象。
    for name in sorted(grads):
        f = lambda _: model.loss(X, y)[0]
        grad_num = eval_numerical_gradient(f, model.params[name], verbose=False, h=1e-5)
        print(f"{name} relative error: {rel_error(grad_num, grads[name])}")

另一个合理性检查是确认网络能在一个只有 50 张图片的小数据集上过拟合。首先尝试一个三层网络，每个隐藏层有 100 个单元。在下面的单元中，调整**学习率**和**权重初始化尺度**，让模型在 20 个 epoch 内过拟合并达到 100% 的训练准确率。

In [ ]:
# TODO：只调整学习率和初始化尺度，让一个三层网络过拟合 50 个训练样本。
#

num_train = 50
small_data = {
  "X_train": data["X_train"][:num_train],
  "y_train": data["y_train"][:num_train],
  "X_val": data["X_val"],
  "y_val": data["y_val"],
}

weight_scale = 1e-2   # 可以尝试修改这里。
learning_rate = 1e-4  # 可以尝试修改这里。


model = FullyConnectedNet(
    [100, 100],
    weight_scale=weight_scale,
    dtype=np.float64
)
solver = Solver(
    model,
    small_data,
    print_every=10,
    num_epochs=20,
    batch_size=25,
    update_rule="sgd",
    optim_config={"learning_rate": learning_rate},
)
solver.train()

plt.plot(solver.loss_history)
plt.title("Training loss history")
plt.xlabel("Iteration")
plt.ylabel("Training loss")
plt.grid(linestyle='--', linewidth=0.5)
plt.show()

现在尝试使用一个五层网络，每层有 100 个单元，在 50 个训练样本上过拟合。同样，你需要调整学习率和权重初始化尺度，但应该能够在 20 个 epoch 内达到 100% 的训练准确率。

In [ ]:
# TODO：只调整学习率和初始化尺度，让一个五层网络过拟合 50 个训练样本。
#

num_train = 50
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}

learning_rate = 2e-3  # 可以尝试修改这里。
weight_scale = 1e-5   # 可以尝试修改这里。


model = FullyConnectedNet(
    [100, 100, 100, 100],
    weight_scale=weight_scale,
    dtype=np.float64
)
solver = Solver(
    model,
    small_data,
    print_every=10,
    num_epochs=20,
    batch_size=25,
    update_rule='sgd',
    optim_config={'learning_rate': learning_rate},
)
solver.train()

plt.plot(solver.loss_history)
plt.title('Training loss history')
plt.xlabel('Iteration')
plt.ylabel('Training loss')
plt.grid(linestyle='--', linewidth=0.5)
plt.show()

## 内联问题 1：
你是否注意到三层网络和五层网络在训练难度上的差异？特别是根据你的实验经验，哪个网络似乎对初始化尺度更敏感？你认为为什么会这样？

## 回答：
[在此填写]

# 更新规则
到目前为止，我们一直使用普通的随机梯度下降（SGD）作为更新规则。更复杂的更新规则可以让深度网络更容易训练。接下来我们将实现几种最常用的更新规则，并将它们与普通 SGD 进行比较。

## SGD+Momentum
带动量的随机梯度下降是一种广泛使用的更新规则，通常能让深度网络比普通随机梯度下降收敛得更快。更多信息请参考 http://cs231n.github.io/neural-networks-3/#sgd 中的 Momentum Update 部分。

打开 `cs231n/optim.py` 文件，阅读文件开头的文档，确保你理解 API。然后在 `sgd_momentum` 函数中实现 SGD+momentum 更新规则，并运行下面的代码检查你的实现。你应该看到小于 `1e-8` 的误差。

In [ ]:
from cs231n.optim import sgd_momentum

N, D = 4, 5
w = np.linspace(-0.4, 0.6, num=N*D).reshape(N, D)
dw = np.linspace(-0.6, 0.4, num=N*D).reshape(N, D)
v = np.linspace(0.6, 0.9, num=N*D).reshape(N, D)

config = {"learning_rate": 1e-3, "velocity": v}
next_w, _ = sgd_momentum(w, dw, config=config)

expected_next_w = np.asarray([
  [ 0.1406,      0.20738947,  0.27417895,  0.34096842,  0.40775789],
  [ 0.47454737,  0.54133684,  0.60812632,  0.67491579,  0.74170526],
  [ 0.80849474,  0.87528421,  0.94207368,  1.00886316,  1.07565263],
  [ 1.14244211,  1.20923158,  1.27602105,  1.34281053,  1.4096    ]])
expected_velocity = np.asarray([
  [ 0.5406,      0.55475789,  0.56891579, 0.58307368,  0.59723158],
  [ 0.61138947,  0.62554737,  0.63970526,  0.65386316,  0.66802105],
  [ 0.68217895,  0.69633684,  0.71049474,  0.72465263,  0.73881053],
  [ 0.75296842,  0.76712632,  0.78128421,  0.79544211,  0.8096    ]])

# 你应该看到相对误差约为 e-8 或更小。
print("next_w error: ", rel_error(next_w, expected_next_w))
print("velocity error: ", rel_error(expected_velocity, config["velocity"]))

完成后，运行下面的代码，分别用 SGD 和 SGD+momentum 训练一个六层网络。你应该会看到 SGD+momentum 收敛得更快。

In [ ]:
num_train = 4000
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}

solvers = {}

for update_rule in ['sgd', 'sgd_momentum']:
    print('Running with ', update_rule)
    model = FullyConnectedNet(
        [100, 100, 100, 100, 100],
        weight_scale=5e-2
    )

    solver = Solver(
        model,
        small_data,
        num_epochs=5,
        batch_size=100,
        update_rule=update_rule,
        optim_config={'learning_rate': 5e-3},
        verbose=True,
    )
    solvers[update_rule] = solver
    solver.train()

fig, axes = plt.subplots(3, 1, figsize=(15, 15))

axes[0].set_title('Training loss')
axes[0].set_xlabel('Iteration')
axes[1].set_title('Training accuracy')
axes[1].set_xlabel('Epoch')
axes[2].set_title('Validation accuracy')
axes[2].set_xlabel('Epoch')

for update_rule, solver in solvers.items():
    axes[0].plot(solver.loss_history, label=f"loss_{update_rule}")
    axes[1].plot(solver.train_acc_history, label=f"train_acc_{update_rule}")
    axes[2].plot(solver.val_acc_history, label=f"val_acc_{update_rule}")

for ax in axes:
    ax.legend(loc="best", ncol=4)
    ax.grid(linestyle='--', linewidth=0.5)

plt.show()

## RMSProp 和 Adam
RMSProp [1] 和 Adam [2] 是两种更新规则，它们通过梯度二阶矩的滑动平均为每个参数设置不同的学习率。

在 `cs231n/optim.py` 文件中，在 `rmsprop` 函数里实现 RMSProp 更新规则，在 `adam` 函数里实现 Adam 更新规则，并使用下面的测试检查实现。

**注意：** 请实现_完整的_ Adam 更新规则（包含偏差校正机制），不要实现课程笔记中最先提到的简化版本。

[1] Tijmen Tieleman and Geoffrey Hinton. "Lecture 6.5-rmsprop: Divide the gradient by a running average of its recent magnitude." COURSERA: Neural Networks for Machine Learning 4 (2012).

[2] Diederik Kingma and Jimmy Ba, "Adam: A Method for Stochastic Optimization", ICLR 2015.

In [ ]:
# 测试 RMSProp 实现
from cs231n.optim import rmsprop

N, D = 4, 5
w = np.linspace(-0.4, 0.6, num=N*D).reshape(N, D)
dw = np.linspace(-0.6, 0.4, num=N*D).reshape(N, D)
cache = np.linspace(0.6, 0.9, num=N*D).reshape(N, D)

config = {'learning_rate': 1e-2, 'cache': cache}
next_w, _ = rmsprop(w, dw, config=config)

expected_next_w = np.asarray([
  [-0.39223849, -0.34037513, -0.28849239, -0.23659121, -0.18467247],
  [-0.132737,   -0.08078555, -0.02881884,  0.02316247,  0.07515774],
  [ 0.12716641,  0.17918792,  0.23122175,  0.28326742,  0.33532447],
  [ 0.38739248,  0.43947102,  0.49155973,  0.54365823,  0.59576619]])
expected_cache = np.asarray([
  [ 0.5976,      0.6126277,   0.6277108,   0.64284931,  0.65804321],
  [ 0.67329252,  0.68859723,  0.70395734,  0.71937285,  0.73484377],
  [ 0.75037008,  0.7659518,   0.78158892,  0.79728144,  0.81302936],
  [ 0.82883269,  0.84469141,  0.86060554,  0.87657507,  0.8926    ]])

# 你应该看到相对误差约为 e-7 或更小。
print('next_w error: ', rel_error(expected_next_w, next_w))
print('cache error: ', rel_error(expected_cache, config['cache']))

In [ ]:
# 测试 Adam 实现
from cs231n.optim import adam

N, D = 4, 5
w = np.linspace(-0.4, 0.6, num=N*D).reshape(N, D)
dw = np.linspace(-0.6, 0.4, num=N*D).reshape(N, D)
m = np.linspace(0.6, 0.9, num=N*D).reshape(N, D)
v = np.linspace(0.7, 0.5, num=N*D).reshape(N, D)

config = {'learning_rate': 1e-2, 'm': m, 'v': v, 't': 5}
next_w, _ = adam(w, dw, config=config)

expected_next_w = np.asarray([
  [-0.40094747, -0.34836187, -0.29577703, -0.24319299, -0.19060977],
  [-0.1380274,  -0.08544591, -0.03286534,  0.01971428,  0.0722929],
  [ 0.1248705,   0.17744702,  0.23002243,  0.28259667,  0.33516969],
  [ 0.38774145,  0.44031188,  0.49288093,  0.54544852,  0.59801459]])
expected_v = np.asarray([
  [ 0.69966,     0.68908382,  0.67851319,  0.66794809,  0.65738853,],
  [ 0.64683452,  0.63628604,  0.6257431,   0.61520571,  0.60467385,],
  [ 0.59414753,  0.58362676,  0.57311152,  0.56260183,  0.55209767,],
  [ 0.54159906,  0.53110598,  0.52061845,  0.51013645,  0.49966,   ]])
expected_m = np.asarray([
  [ 0.48,        0.49947368,  0.51894737,  0.53842105,  0.55789474],
  [ 0.57736842,  0.59684211,  0.61631579,  0.63578947,  0.65526316],
  [ 0.67473684,  0.69421053,  0.71368421,  0.73315789,  0.75263158],
  [ 0.77210526,  0.79157895,  0.81105263,  0.83052632,  0.85      ]])

# 你应该看到相对误差约为 e-7 或更小。
print('next_w error: ', rel_error(expected_next_w, next_w))
print('v error: ', rel_error(expected_v, config['v']))
print('m error: ', rel_error(expected_m, config['m']))

调试好 RMSProp 和 Adam 的实现后，运行下面的代码，用这些新的更新规则训练一组深层网络：

In [ ]:
learning_rates = {'rmsprop': 1e-4, 'adam': 1e-3}
for update_rule in ['adam', 'rmsprop']:
    print('Running with ', update_rule)
    model = FullyConnectedNet(
        [100, 100, 100, 100, 100],
        weight_scale=5e-2
    )
    solver = Solver(
        model,
        small_data,
        num_epochs=5,
        batch_size=100,
        update_rule=update_rule,
        optim_config={'learning_rate': learning_rates[update_rule]},
        verbose=True
    )
    solvers[update_rule] = solver
    solver.train()
    print()

fig, axes = plt.subplots(3, 1, figsize=(15, 15))

axes[0].set_title('Training loss')
axes[0].set_xlabel('Iteration')
axes[1].set_title('Training accuracy')
axes[1].set_xlabel('Epoch')
axes[2].set_title('Validation accuracy')
axes[2].set_xlabel('Epoch')

for update_rule, solver in solvers.items():
    axes[0].plot(solver.loss_history, label=f"{update_rule}")
    axes[1].plot(solver.train_acc_history, label=f"{update_rule}")
    axes[2].plot(solver.val_acc_history, label=f"{update_rule}")

for ax in axes:
    ax.legend(loc='best', ncol=4)
    ax.grid(linestyle='--', linewidth=0.5)

plt.show()

## 内联问题 2：

AdaGrad 和 Adam 一样，也是按参数自适应的优化方法，它使用如下更新规则：

```
cache += dw**2
w += - learning_rate * dw / (np.sqrt(cache) + eps)
```

John 注意到，当他用 AdaGrad 训练网络时，更新量会变得非常小，网络学习得很慢。根据你对 AdaGrad 更新规则的理解，为什么更新量会变得很小？Adam 会有同样的问题吗？

## 回答：
[在此填写]

# 训练一个好模型
在 CIFAR-10 上训练你能得到的最好的全连接模型，并把最佳模型存入 `best_model` 变量。我们要求使用全连接网络在验证集上至少达到 50% 的准确率。

如果调参足够仔细，达到 55% 以上是可能的；但这一部分不强制要求，也不会因此给额外分。下一次作业会要求你在 CIFAR-10 上训练尽可能好的卷积网络，我们更希望你把精力投入到卷积网络上，而不是继续深挖全连接网络。

**注意：** 在下一次作业中，你会学习 BatchNormalization 和 Dropout 等技术，它们可以帮助训练更强的模型。

In [ ]:
best_model = None

################################################################################
# TODO：在 CIFAR-10 上训练你能得到的最佳 FullyConnectedNet。你可能会发现    #
# batch/layer normalization 和 dropout 很有用。将你的最佳模型存入 best_model。 #                                                     #
################################################################################
# *****你的代码开始 (不要删除或修改这一行)*****



# *****你的代码结束 (不要删除或修改这一行)*****
################################################################################
#                              你的代码结束                                #
################################################################################

# 测试你的模型
在验证集和测试集上运行你的最佳模型。你应该在验证集和测试集上都达到至少 50% 的准确率。

In [ ]:
y_test_pred = np.argmax(best_model.loss(data['X_test']), axis=1)
y_val_pred = np.argmax(best_model.loss(data['X_val']), axis=1)
print('Validation set accuracy: ', (y_val_pred == data['y_val']).mean())
print('Test set accuracy: ', (y_test_pred == data['y_test']).mean())